# 01A — Check BMI imputation


# Summary section

**Motivation:** 
`osteo_impute_final.csv` ships as a five-imputation stack (m = 1–5) with no
`m=0`, so the original un-imputed BMI column is not in the file. This notebook
recovers the missingness structure indirectly and checks that the stack behaves
the way the `do-file` implies it should.

**What it does**

1. Recovers which BMI values were observed and which were imputed, using
   within-patient variance across the five imputations: a value that varies was
   missing, a value that is constant was observed.
2. Runs integrity checks on the stack — imputation count, whether
   `Strata` + `Group` uniquely identifies a patient within an imputation
3. Pools case and control BMI means under Rubin's rules, separating
   within- from between-imputation variance
4. Compares BMI means on the observed-only subset by group.

**What it finds**

- BMI is **missing for 68% of patients**
- the missingness is strongly associated with the outcome: 13% of cases had an observed BMI against 51% of controls.
The imputation was therefore doing most of its work on the case side, which is
where the pooled case mean moves.

# 1. Which BMI values were imputed

The stack runs 1–5 with no `m=0`, so the original un-imputed copy is not in
this file. It can be recovered indirectly: a value that varies across the five
imputations was missing; a value that is constant was observed.

This matters for reporting the missingness rate and for building a
complete-case comparison.

In [1]:
from pathlib import Path

import pandas as pd

# Resolve paths relative to the project root so the notebook runs from anywhere
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SOURCE_FILE = DATA_DIR / "osteo_impute_final.csv"

# Published reference values, for reconciliation
PUBLISHED_N_CASES = 30_517
PUBLISHED_N_TOTAL = PUBLISHED_N_CASES * 2  # 1:1 matching

assert SOURCE_FILE.exists(), f"Not found: {SOURCE_FILE}"
print(f"Source: {SOURCE_FILE}")
print(f"Size:   {SOURCE_FILE.stat().st_size / 1024**2:.0f} MiB")

Source: /Users/thanhbrown/Home/work_portfolio/osteo_ml/data/osteo_impute_final.csv
Size:   157 MiB


In [2]:
bmi = pd.read_csv(
    SOURCE_FILE,
    usecols=["_Imputation_", "Strata", "Group", "BMI_Avg", 
             "BMI_Avg_Prior", "BMI_Avg_Cat", "BMI_Avg_Cat_Num", 
             "BMI_Avg_Prior_Cat", "BMI_Avg_Prior_Cat_Num"],
    low_memory=False,
)
bmi.head()


,_Imputation_,Group,BMI_Avg,BMI_Avg_Prior,BMI_Avg_Cat,BMI_Avg_Cat_Num,BMI_Avg_Prior_Cat,BMI_Avg_Prior_Cat_Num,Strata
0,1,Control,37.834028,NaN,>30,3.0,NaN,NaN,9576
1,1,Control,32.948625,NaN,>30,3.0,NaN,NaN,13833
2,1,Control,33.388021,NaN,>30,3.0,NaN,NaN,28858
3,1,Control,28.308928,28.232490,25-30,2.0,25-30,2.0,16496
4,1,Case,22.666667,18.086505,<25,1.0,<25,1.0,1


In [3]:
bmi.shape

(305180, 9)

In [4]:
bmi_cols = bmi.filter(regex='^BMI').columns

bmi.filter(regex='^BMI').isna().sum()

BMI_Avg                       0
BMI_Avg_Prior            220025
BMI_Avg_Cat              207100
BMI_Avg_Cat_Num          207100
BMI_Avg_Prior_Cat        220025
BMI_Avg_Prior_Cat_Num    220025
dtype: int64

In [5]:
spread = bmi.groupby(["Strata", "Group"])["BMI_Avg"].nunique()
n_imputed = (spread > 1).sum()
n_total = spread.size

print(f"Imputed:  {n_imputed:,} / {n_total:,} ({n_imputed / n_total:.1%})")
print(f"Observed: {n_total - n_imputed:,}")

Imputed:  41,496 / 61,034 (68.0%)
Observed: 19,538


In [6]:
# Prevent Pandas from truncating long lists or text in output cells
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)

# Round values to 2 decimal places and collect into lists
bmi.groupby(["Strata", "Group"])["BMI_Avg"].apply(
    lambda x: x.round(2).tolist()
)

Strata  Group  
1       Case       [22.67, 22.67, 22.67, 22.67, 22.67]
        Control     [27.6, 26.47, 24.87, 32.97, 28.99]
2       Case       [32.63, 33.61, 19.65, 22.68, 31.82]
        Control    [21.52, 21.52, 21.52, 21.52, 21.52]
3       Case        [32.92, 31.97, 21.1, 35.42, 37.32]
                                  ...                 
30515   Control    [34.53, 21.32, 38.08, 32.22, 28.71]
30516   Case       [29.87, 29.91, 25.36, 23.54, 24.92]
        Control    [35.46, 24.12, 35.54, 27.84, 17.79]
30517   Case       [14.38, 20.84, 24.26, 31.78, 30.57]
        Control     [21.58, 25.03, 22.0, 26.94, 28.31]
Name: BMI_Avg, Length: 61034, dtype: object

In [7]:
# 1. Confirm 5 imputations, no m=0 stratum
bmi["_Imputation_"].value_counts().sort_index()

_Imputation_
1    61036
2    61036
3    61036
4    61036
5    61036
Name: count, dtype: int64

In [8]:
# 2. Patient key: Strata + Group should be unique within an imputation
m1 = bmi[bmi["_Imputation_"] == 1]
m1.duplicated(subset=["Strata", "Group"]).sum()   # expect 0
m1.groupby("Strata").size().value_counts()        # expect all 2


2    30515
3        2
Name: count, dtype: int64

> 30,515 strata have 2 rows and 2 strata have 3 rows (will investigate these 2 strata in the next notebook

In [9]:
# 3. Recover which patients were actually observed:
#    zero within-patient variance across imputations = observed
var = bmi.groupby(["Strata", "Group"])["BMI_Avg"].var()
(var == 0).sum()          # should be ~19,538

np.int64(19538)

In [10]:
# 4. Did the do-file's outlier trim survive?
bmi["BMI_Avg"].describe()   # expect min > 11, max < 100

count    305180.000000
mean         27.700615
std           6.992462
min          11.000409
25%          22.919192
50%          27.283616
75%          32.037607
max          96.150000
Name: BMI_Avg, dtype: float64

# 2. Check Rubin pool

In [11]:
import numpy as np
import pandas as pd

M = bmi["_Imputation_"].nunique()          # 5

rows = {}
for grp, g in bmi.groupby("Group"):
    per_m = g.groupby("_Imputation_")["BMI_Avg"].agg(["mean", "sem", "count"])
    Qbar = per_m["mean"].mean()            # pooled estimate
    Ubar = (per_m["sem"] ** 2).mean()      # within-imputation variance
    B    = per_m["mean"].var(ddof=1)       # between-imputation variance
    T    = Ubar + (1 + 1/M) * B            # Rubin total variance

    r    = (1 + 1/M) * B / Ubar            # relative increase in variance
    lam  = (1 + 1/M) * B / T               # fraction of missing information

    rows[grp] = {
        "n_per_imp": int(per_m["count"].iloc[0]),
        "mean": Qbar,
        "SE": np.sqrt(T),
        "SE_within": np.sqrt(Ubar),
        "SE_between": np.sqrt(B),
        "r": r,
        "FMI": lam,
    }

pd.DataFrame(rows).T.round(4)

,n_per_imp,mean,SE,SE_within,SE_between,r,FMI
Case,30518.0,27.6307,0.0629,0.0403,0.0441,1.4351,0.5893
Control,30518.0,27.7706,0.0580,0.0397,0.0386,1.1337,0.5313


So for `cases`, roughly 59% of the uncertainty in the pooled mean comes from the five imputations disagreeing with each other, not from sampling noise within any one of them. `SE_between` (0.0441) actually exceeds `SE_within` (0.0403) — **the imputation model is contributing more uncertainty than the data does**.

## 2.1 Conclusion — BMI pooling
- Since the published paper mentioned that "unadjusted ORs matching to two decimals", meaning adding BMI to the model didn't change the other coefficients at two decimal places. That's the direct evidence that BMI adjustment isn't load-bearing — the OR of 0.979 on its own only tells you BMI is weakly associated with the outcome

> BMI barely affects the published estimates, therefore however the imputation behaves, the reproduction target (OR 3.970 for chronic hyponatremia) shouldn't move.

# 3. BMI missingness is strongly outcome-associated

In [12]:
# Take observed-only subset, "observed" here means had BMI data available
var = bmi.groupby(["Strata", "Group"])["BMI_Avg"].var()
observed = var[var == 0].index

m1 = bmi[bmi["_Imputation_"] == 1].set_index(["Strata", "Group"])

obs = m1.loc[observed].groupby("Group")["BMI_Avg"].agg(["mean", "count"])
obs["observed_rate"]    = obs["count"] / m1.groupby("Group").size() * 100

obs.round(2)

,mean,count,observed_rate
Group,,,
Case,26.95,4027,13.20
Control,28.28,15511,50.83


Only 13% (4,027 / 30,518 = 13.2%) of cases had a real BMI on record; 51% (15,511 / 30,518 = 50.8%) of controls did. Controls were nearly four times more likely to have a BMI on record than cases (50.83 / 13.20 = 3.85).

| Group | observed only means | pooled means (All Patients) |
| :--- | :--- |:--- |
| **Case** | 26.95 | 27.63 |
| **Control** | 28.28 | 27.77 |

It also means the imputation was doing most of its work on the case side specifically, which is why the case mean rose 0.68 (26.95 → 27.63) and the control mean fell 0.51 (28.28 → 27.77)